In [1]:
from __future__ import annotations
# Performance Analytics: Top-5 funds vs Nifty 50 & Nifty 100 over ~3 years + tracking error
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HERE = Path(__file__).resolve() if '__file__' in globals() else Path.cwd()

def _find_repo_root(start: Path) -> Path:
    cand = start
    for _ in range(18):
        if (cand / 'Data' / 'processed' / 'fund_scorecard.csv').exists() and (cand / 'Data' / 'processed' / 'benchmark_indices_clean.csv').exists():
            return cand
        if cand.name == 'notebooks':
            parent = cand.parent
            if (parent / 'Data' / 'processed' / 'fund_scorecard.csv').exists() and (parent / 'Data' / 'processed' / 'benchmark_indices_clean.csv').exists():
                return parent
        cand = cand.parent
    return start.parent

REPO_ROOT = _find_repo_root(HERE)
DATA_DIR = REPO_ROOT / 'Data' / 'processed'

print('REPO_ROOT:', REPO_ROOT.resolve())
print('DATA_DIR:', DATA_DIR.resolve())


REPO_ROOT: C:\Mutual Fund Analytics
DATA_DIR: C:\Mutual Fund Analytics\Data\processed


In [2]:
fund_scorecard_path = DATA_DIR / 'fund_scorecard.csv'
daily_returns_path = DATA_DIR / 'daily_returns_all_schemes.csv'
bench_path = DATA_DIR / 'benchmark_indices_clean.csv'
for p in [fund_scorecard_path, daily_returns_path, bench_path]:
    if not p.exists():
        raise FileNotFoundError(f'Missing required file: {p.resolve()}')
print('Loaded paths OK')


Loaded paths OK


In [3]:
def require_columns(df: pd.DataFrame, required: set[str], df_name: str) -> None:
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'{df_name} missing columns: {sorted(missing)}')

def load_csv(path: Path, df_name: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    if df is None or df.empty:
        raise ValueError(f'{df_name} is empty: {path.resolve()}')
    return df


In [4]:
fund_scorecard_df = load_csv(fund_scorecard_path, 'fund_scorecard.csv')
require_columns(fund_scorecard_df, {'amfi_code','scheme_name','fund_score_0_100'}, 'fund_scorecard.csv')
fund_scorecard_df['amfi_code'] = pd.to_numeric(fund_scorecard_df['amfi_code'], errors='coerce').astype('Int64')
fund_scorecard_df['fund_score_0_100'] = pd.to_numeric(fund_scorecard_df['fund_score_0_100'], errors='coerce')
fund_scorecard_df = fund_scorecard_df.dropna(subset=['amfi_code','fund_score_0_100']).copy()
if fund_scorecard_df.empty:
    raise ValueError('fund_scorecard.csv has no rows after cleaning.')
fund_scorecard_df = fund_scorecard_df.sort_values('fund_score_0_100', ascending=False).reset_index(drop=True)
top_k = 5
top_funds = fund_scorecard_df.head(top_k).copy()
print('Top funds:')
print(top_funds[['amfi_code','scheme_name','fund_score_0_100']].to_string(index=False))


Top funds:
 amfi_code                                   scheme_name  fund_score_0_100
    119599     SBI Small Cap Fund - Direct Plan - Growth         77.051282
    120842 Kotak Emerging Equity Fund - Regular - Growth         75.000000
    101207        ABSL Small Cap Fund - Regular - Growth         74.871795
    120843        Kotak Flexicap Fund - Regular - Growth         73.717949
    119598    SBI Small Cap Fund - Regular Plan - Growth         69.102564


In [5]:
daily_returns_df = load_csv(daily_returns_path, 'daily_returns_all_schemes.csv')
require_columns(daily_returns_df, {'date','amfi_code','daily_return'}, 'daily_returns_all_schemes.csv')
daily_returns_df['date'] = pd.to_datetime(daily_returns_df['date'], errors='coerce')
daily_returns_df['amfi_code'] = pd.to_numeric(daily_returns_df['amfi_code'], errors='coerce').astype('Int64')
daily_returns_df['daily_return'] = pd.to_numeric(daily_returns_df['daily_return'], errors='coerce')
if 'scheme_name' in daily_returns_df.columns:
    daily_returns_df['scheme_name'] = daily_returns_df['scheme_name']
daily_returns_df = daily_returns_df.dropna(subset=['date','amfi_code','daily_return']).copy()
if daily_returns_df.empty:
    raise ValueError('daily_returns_all_schemes.csv has no valid rows after cleaning.')


In [6]:
bench_df = load_csv(bench_path, 'benchmark_indices_clean.csv')
require_columns(bench_df, {'date','index_name','close_value'}, 'benchmark_indices_clean.csv')
bench_df['date'] = pd.to_datetime(bench_df['date'], errors='coerce')
bench_df['index_name'] = bench_df['index_name'].astype(str)
bench_df['close_value'] = pd.to_numeric(bench_df['close_value'], errors='coerce')
bench_df = bench_df.dropna(subset=['date','index_name','close_value']).copy()
idx_lower = bench_df['index_name'].str.lower()
mask_50 = idx_lower.str.contains('nifty') & idx_lower.str.contains('50')
mask_100 = idx_lower.str.contains('nifty') & idx_lower.str.contains('100')
bench_nifty50 = bench_df.loc[mask_50, ['date','index_name','close_value']].copy()
bench_nifty100 = bench_df.loc[mask_100, ['date','index_name','close_value']].copy()
if bench_nifty50.empty:
    raise ValueError('Could not find Nifty 50 in benchmark_indices_clean.csv')
if bench_nifty100.empty:
    raise ValueError('Could not find Nifty 100 in benchmark_indices_clean.csv')
bench_nifty50 = bench_nifty50.sort_values('date').copy()
bench_nifty100 = bench_nifty100.sort_values('date').copy()
bench_nifty50['benchmark_return'] = bench_nifty50['close_value'].pct_change()
bench_nifty100['benchmark_return'] = bench_nifty100['close_value'].pct_change()
bench_nifty50 = bench_nifty50.dropna(subset=['benchmark_return']).copy()
bench_nifty100 = bench_nifty100.dropna(subset=['benchmark_return']).copy()


In [7]:
fund_max_end = daily_returns_df['date'].max()
bench_max_end = min(bench_nifty50['date'].max(), bench_nifty100['date'].max())
end_date = min(fund_max_end, bench_max_end)
start_date = end_date - pd.Timedelta(days=int(round(3 * 365.25)))
print('Window start:', start_date.date(), 'end:', end_date.date())
b50 = bench_nifty50[(bench_nifty50['date'] >= start_date) & (bench_nifty50['date'] <= end_date)].copy()
b100 = bench_nifty100[(bench_nifty100['date'] >= start_date) & (bench_nifty100['date'] <= end_date)].copy()
if b50.empty or b100.empty:
    raise ValueError('Benchmark series empty after applying 3-year window.')


Window start: 2023-05-29 end: 2026-05-29


In [8]:
TRADING_DAYS = 252

def tracking_error(fund_daily: pd.DataFrame, bench_daily: pd.DataFrame) -> float:
    merged = fund_daily.merge(bench_daily[['date','benchmark_return']], on='date', how='inner')
    if len(merged) < 2:
        return float('nan')
    diff = (merged['daily_return'] - merged['benchmark_return']).astype(float).values
    return float(np.std(diff, ddof=1) * np.sqrt(TRADING_DAYS))

def cumulative_from_returns(returns: pd.Series) -> pd.Series:
    return (1.0 + returns.fillna(0.0)).cumprod() - 1.0


In [9]:
fund_returns_window = daily_returns_df[(daily_returns_df['date'] >= start_date) & (daily_returns_df['date'] <= end_date)].copy()
rows = []
for _, r in top_funds.iterrows():
    code = int(r['amfi_code'])
    name = str(r['scheme_name'])
    fr = fund_returns_window[fund_returns_window['amfi_code'] == code].copy().sort_values('date')
    if fr.empty:
        continue
    te_50 = tracking_error(fr[['date','daily_return']], b50)
    te_100 = tracking_error(fr[['date','daily_return']], b100)
    rows.append({
        'amfi_code': code,
        'scheme_name': name,
        'tracking_error_nifty50': te_50,
        'tracking_error_nifty100': te_100,
        'n_obs_fund': int(fr.shape[0]),
    })

results_df = pd.DataFrame(rows).sort_values(['tracking_error_nifty100'], ascending=True)
print(results_df.to_string(index=False))


 amfi_code                                   scheme_name  tracking_error_nifty50  tracking_error_nifty100  n_obs_fund
    120843        Kotak Flexicap Fund - Regular - Growth                4.518275                 0.206410        1097
    120842 Kotak Emerging Equity Fund - Regular - Growth                4.521993                 0.220315        1097
    119599     SBI Small Cap Fund - Direct Plan - Growth                4.519893                 0.272367        1097
    119598    SBI Small Cap Fund - Regular Plan - Growth                4.523039                 0.286772        1097
    101207        ABSL Small Cap Fund - Regular - Growth                4.521626                 0.292998        1097


In [10]:
plt.figure(figsize=(12, 7))
b50_plot = b50.sort_values('date').copy()
b100_plot = b100.sort_values('date').copy()
b50_plot['cum_return'] = cumulative_from_returns(b50_plot['benchmark_return'])
b100_plot['cum_return'] = cumulative_from_returns(b100_plot['benchmark_return'])
plt.plot(b50_plot['date'], b50_plot['cum_return'] * 100.0, label='Nifty 50', linewidth=2.5, color='black')
plt.plot(b100_plot['date'], b100_plot['cum_return'] * 100.0, label='Nifty 100', linewidth=2.5, color='gray')
for _, r in top_funds.iterrows():
    code = int(r['amfi_code'])
    name = str(r['scheme_name'])
    fr = fund_returns_window[fund_returns_window['amfi_code'] == code].copy().sort_values('date')
    if fr.empty:
        continue
    fr['cum_return'] = cumulative_from_returns(fr['daily_return'])
    plt.plot(fr['date'], fr['cum_return'] * 100.0, label=f'{name} ({code})', linewidth=1.8, alpha=0.95)
plt.title('Top-5 Fund Performance vs Nifty 50 & Nifty 100 (~3 years)')
plt.xlabel('Date')
plt.ylabel('Cumulative return (%)')
plt.grid(True, alpha=0.25)
plt.legend(fontsize=9, loc='best')

out_png = DATA_DIR / 'benchmark_comparison_top5_funds_nifty50_nifty100.png'
plt.tight_layout()
plt.savefig(out_png, dpi=200)
plt.close()
print('Wrote chart PNG:', out_png.resolve())


Wrote chart PNG: C:\Mutual Fund Analytics\Data\processed\benchmark_comparison_top5_funds_nifty50_nifty100.png


In [11]:
out_te = DATA_DIR / 'tracking_error_top5_funds_vs_benchmarks.csv'
results_df.to_csv(out_te, index=False)
print('Wrote tracking error table:', out_te.resolve())


Wrote tracking error table: C:\Mutual Fund Analytics\Data\processed\tracking_error_top5_funds_vs_benchmarks.csv
